# 05_customer_segmentation
Customer segmentation using RFM analysis and K-Means clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../data/processed/superstore_features.csv")
df["Order_Date"] = pd.to_datetime(df["Order_Date"])


## Build RFM Table

In [ ]:
snapshot_date = df["Order_Date"].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby("Customer_ID")
      .agg(
          Recency=("Order_Date", lambda x: (snapshot_date - x.max()).days),
          Frequency=("Order_ID", "nunique"),
          Monetary=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Avg_Order_Value=("Sales", "mean")
      )
      .reset_index()

)

display(rfm.head())


## RFM Scores

In [ ]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1])
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5])

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

display(rfm.head())


## Rule-Based Customer Segments

In [ ]:
def segment(row):
    r = int(row["R_Score"])
    f = int(row["F_Score"])
    m = int(row["M_Score"])

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal_Customers"
    elif r >= 4 and f <= 2:
        return "Potential_Loyalists"
    elif r <= 2 and f >= 3:
        return "At_Risk"
    elif r == 1 and f <= 2:
        return "Lost_Customers"
    else:
        return "Needs_Attention"

rfm["Segment"] = rfm.apply(segment, axis=1)

display(rfm["Segment"].value_counts())


## K-Means Segmentation

In [ ]:
features = rfm[["Recency","Frequency","Monetary"]]

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

scores = {}

for k in range(2,9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(scaled)
    scores[k] = silhouette_score(scaled, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(scaled)


## Cluster Summary

In [ ]:
cluster_summary = (
    rfm.groupby("Cluster")
       .agg(
           Customers=("Customer_ID","count"),
           Avg_Recency=("Recency","mean"),
           Avg_Frequency=("Frequency","mean"),
           Avg_Monetary=("Monetary","mean"),
           Avg_Profit=("Profit","mean")
       )
)

display(cluster_summary)


## Visualizations

In [ ]:
plt.figure(figsize=(8,5))
rfm["Segment"].value_counts().plot(kind="bar")
plt.title("Customer_Segments")
plt.show()

plt.figure(figsize=(8,6))
plt.scatter(
    rfm["Frequency"],
    rfm["Monetary"],
    c=rfm["Cluster"]
)
plt.xlabel("Frequency")
plt.ylabel("Monetary")
plt.title("Customer_Clusters")
plt.show()


## Export Results

In [ ]:
rfm.to_csv("../data/processed/customer_segments.csv", index=False)
print("Customer segments exported.")


## Business Insights

Complete this section after reviewing the results.

- Which segment generates the highest revenue?
- Which customers are at risk?
- How many Champions are there?
- Which cluster should receive premium offers?
- Which cluster should receive retention campaigns?
- Which cluster is suitable for upselling?
